# Civil War News — Notebook 1: Data Preprocessing

Loads the AmericanStories 1860–1865 corpus, applies OCR correction, computes
LSD and VADER sentiment scores, and saves the result for Notebooks 2 and 3.

**Estimated run time:** 4–8 hours (OCR correction + LSD/VADER are CPU-bound; no GPU needed).

**Storage note (USE_DRIVE=False):** The processed dataset is stored in Colab's
ephemeral `/content/` directory and is **lost when the session ends**. Run
Notebooks 2 and 3 in the **same session**, or set `USE_DRIVE=True` to persist
across sessions.

**Outputs:** `news_proc/` — a HuggingFace dataset with OCR-corrected text
and LSD/VADER sentiment scores, saved to `BASE_PATH`.

## Section 1: Install Packages

In [ ]:
%pip install --q \
    datasets \
    openpyxl \
    pandas \
    symspellpy \
    vaderSentiment \
    huggingface_hub \
    git+https://github.com/Pleias/OCRoscope.git

print('Packages installed')

## Section 2: Configuration

Edit `USE_DRIVE` and (if True) `DRIVE_BASE` before running.


In [ ]:
import os

# ======================================================================
# STORAGE
USE_DRIVE        = True   # True: Google Drive (Colab, persistent); False: ephemeral /content/
USE_LOCAL_CORPUS = True   # True: skip HF download and load corpus from CORPUS_PATH
DRIVE_BASE       = '/content/drive/MyDrive/CivilWarNews'
LOCAL_BASE       = '/content/data'

HF_REPO          = 'patrickjcrawford/civil-war-news'   # private dataset repo for outputs

# ======================================================================
# SENTIMENT (LSD + VADER — computed here; transformer models run in Notebook 2)
RUN_LSD   = True   # Lexicoder Sentiment Dictionary
RUN_VADER = True   # VADER lexicon

# ======================================================================
# (No edits needed below this line)
BASE_PATH      = DRIVE_BASE if USE_DRIVE else LOCAL_BASE
LCCN_META_PATH = f'{BASE_PATH}/chronicling-america-meta.xlsx'
LSD_PATH       = f'{BASE_PATH}/lsd_texts'
NEWS_PROC_PATH = f'{BASE_PATH}/news_proc'
# Corpus is always extracted to ephemeral storage — large and re-extractable from Drive zip
CORPUS_PATH    = f'{LOCAL_BASE}/americanstories_civilwar'

# makedirs is deferred to Section 3 (after Drive is mounted when USE_DRIVE=True)
print(f'Storage  : {"Google Drive" if USE_DRIVE else os.path.abspath(BASE_PATH)}')
print(f'Output   : {NEWS_PROC_PATH}')
print(f'Corpus   : {CORPUS_PATH} (ephemeral)')
print(f'Sentiment: LSD={RUN_LSD}, VADER={RUN_VADER}')
print(f'HF Repo  : hf://datasets/{HF_REPO}')

## Section 3: Storage Setup

Mounts Google Drive (if `USE_DRIVE=True`) or prompts for file upload.
Required file: `chronicling-america-meta.xlsx` (~50 KB).


In [ ]:
import os
from huggingface_hub import HfApi, login

# ── Hugging Face login ─────────────────────────────────────────────────────
try:
    from google.colab import userdata as _ud
    login(token=_ud.get('HF_TOKEN'), add_to_git_credential=False)
    print('[OK] Logged in to Hugging Face.')
except Exception as _hf_e:
    print(f'HF login: {_hf_e}')
    print('Add HF_TOKEN to Colab Secrets (key icon in sidebar).')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    os.makedirs(BASE_PATH, exist_ok=True)
    os.makedirs(NEWS_PROC_PATH, exist_ok=True)
    os.makedirs(LSD_PATH, exist_ok=True)
    print(f'Drive mounted. Working from: {DRIVE_BASE}')
else:
    os.makedirs(BASE_PATH, exist_ok=True)
    os.makedirs(NEWS_PROC_PATH, exist_ok=True)
    os.makedirs(LSD_PATH, exist_ok=True)
    print('Ephemeral mode — /content/ (lost at session end)')
    print('Run Notebooks 2 and 3 in this same session before disconnecting.')

if not os.path.exists(LCCN_META_PATH):
    print(f'\nMISSING: chronicling-america-meta.xlsx')
    if not USE_DRIVE:
        print('Opening file picker — select chronicling-america-meta.xlsx:')
        from google.colab import files as colab_files
        uploaded = colab_files.upload()
        for fname, data in uploaded.items():
            if fname.endswith('.xlsx'):
                with open(LCCN_META_PATH, 'wb') as fh:
                    fh.write(data)
                print(f'Saved: {LCCN_META_PATH}')
    else:
        raise FileNotFoundError(f'Upload chronicling-america-meta.xlsx to {DRIVE_BASE}')
else:
    print('[OK] chronicling-america-meta.xlsx found')

# LSD files check
if RUN_LSD:
    lsd_required = [
        (f'{LSD_PATH}/lsdSentiment.py',        'lsdSentiment.py'),
        (f'{LSD_PATH}/positive_words.txt',     'positive_words.txt'),
        (f'{LSD_PATH}/negative_words.txt',     'negative_words.txt'),
        (f'{LSD_PATH}/neg_positive_words.txt', 'neg_positive_words.txt'),
        (f'{LSD_PATH}/neg_negative_words.txt', 'neg_negative_words.txt'),
    ]
    missing = [(fp, lb) for fp, lb in lsd_required if not os.path.exists(fp)]
    for fp, lb in lsd_required:
        print(f'  [{"OK" if os.path.exists(fp) else "MISSING"}] {lb}')
    if missing:
        print('\nOpening file picker -- select all lsd_texts files at once:')
        print('  lsdSentiment.py, positive_words.txt, negative_words.txt,')
        print('  neg_positive_words.txt, neg_negative_words.txt')
        from google.colab import files as colab_files
        uploaded = colab_files.upload()
        for fname, data in uploaded.items():
            dest = f'{LSD_PATH}/{fname}'
            with open(dest, 'wb') as fh:
                fh.write(data)
            print(f'  Saved: {dest}')
        still = [lb for fp, lb in lsd_required if not os.path.exists(fp)]
        if still:
            print(f'WARNING: still missing: {still}')
        else:
            print('All LSD files present.')
    else:
        print('All LSD files present.')

## Section 4: LCCN Metadata Filter

Builds a whitelist of Library of Congress Control Numbers (LCCNs) for
English-language newspapers 1860–1865.


In [ ]:
import pandas as pd

meta_df       = pd.read_excel(LCCN_META_PATH)
include_lccns = set(meta_df[meta_df['Languages'].str.strip().eq('English')]['LCCN'].tolist())
print(f'English-language LCCNs in whitelist: {len(include_lccns):,}')

English-language LCCNs in whitelist: 457


## Section 5: Load & Clean

Downloads AmericanStories 1860–1865, applies SymSpell OCR correction, computes
OCR quality metrics, and saves the filtered dataset.

**OCR quality filter (post-cleaning):**
- `ocr_quality_proc >= 95` — at least 95% of word segments recognized
- `nonchar_proc <= 10` — at most 10% non-character tokens

**Alternative OCR approaches (not implemented):**
- *Skip correction*: MacBERTh and bert_1760_1900 were trained on OCR-noisy text
  and may not need correction. Saves several hours.
- *NeusSpell*: Neural contextual spell correction (~5x slower than SymSpell).


In [ ]:
import re, string, importlib.resources
from functools import lru_cache
from symspellpy import SymSpell, Verbosity
from datasets import load_dataset, concatenate_datasets

try:
    from ocroscope import ocr_evaluation
    HAS_OCROSCOPE = True
    print('ocroscope available -- OCR quality filtering enabled')
except ImportError:
    HAS_OCROSCOPE = False
    print('ocroscope not installed -- skipping quality filter')

sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
with importlib.resources.path('symspellpy', 'frequency_dictionary_en_82_765.txt') as d:
    sym_spell.load_dictionary(str(d), term_index=0, count_index=1)
print('SymSpell dictionary loaded')

# ── Unicode typographic ligatures → ASCII ─────────────────────────────────────
_LIGATURE_TABLE = str.maketrans(
    {'ﬁ': 'fi', 'ﬂ': 'fl', 'ﬀ': 'ff', 'ﬃ': 'ffi', 'ﬄ': 'ffl', 'ﬅ': 'st', 'ﬆ': 'st'})

# ── tl/li OCR errors: broken type rendered "th" as "tl" and "h" as "li" ───────
# Constrained to common function words at word boundaries — safe, high yield.
_TL_LI_SUBS = [
    # "the / that / this / there / they / them / their / through"
    (re.compile(r'\btlie\b'),      'the'),    (re.compile(r'\bTlie\b'),      'The'),
    (re.compile(r'\btliat\b'),     'that'),   (re.compile(r'\bTliat\b'),     'That'),
    (re.compile(r'\btliis\b'),     'this'),   (re.compile(r'\bTliis\b'),     'This'),
    (re.compile(r'\btliere\b'),    'there'),  (re.compile(r'\bTliere\b'),    'There'),
    (re.compile(r'\btliey\b'),     'they'),   (re.compile(r'\bTliey\b'),     'They'),
    (re.compile(r'\btliem\b'),     'them'),
    (re.compile(r'\btlieir\b'),    'their'),
    (re.compile(r'\btlirough\b'),  'through'),(re.compile(r'\bTlirough\b'),  'Through'),
    # "what / where / when / which / while / whether"
    (re.compile(r'\bwliat\b'),     'what'),   (re.compile(r'\bWliat\b'),     'What'),
    (re.compile(r'\bwliere\b'),    'where'),  (re.compile(r'\bWliere\b'),    'Where'),
    (re.compile(r'\bwlien\b'),     'when'),   (re.compile(r'\bWlien\b'),     'When'),
    (re.compile(r'\bwliich\b'),    'which'),  (re.compile(r'\bWliich\b'),    'Which'),
    (re.compile(r'\bwliile\b'),    'while'),  (re.compile(r'\bWliile\b'),    'While'),
    (re.compile(r'\bwlietlier\b'), 'whether'),
    # "with / other / has / his / have / had / her"
    (re.compile(r'\bwitli\b'),     'with'),   (re.compile(r'\bWitli\b'),     'With'),
    (re.compile(r'\botlier\b'),    'other'),  (re.compile(r'\bOtlier\b'),    'Other'),
    (re.compile(r'\blias\b'),      'has'),
    (re.compile(r'\bliis\b'),      'his'),
    (re.compile(r'\bliave\b'),     'have'),   (re.compile(r'\bLiave\b'),     'Have'),
    (re.compile(r'\bliad\b'),      'had'),
    (re.compile(r'\blier\b'),      'her'),
]

# ── Whitespace / punctuation ───────────────────────────────────────────────────
RE_DOUBLE_QUOTE     = re.compile(r"''|``")          # two single-quotes or backticks → "
RE_DOT_HYPHEN       = re.compile(r'\.-')
RE_AUD              = re.compile(r'\b[aA]ud\b')
RE_THO              = re.compile(r'\b[tT]ho\b')
RE_ING1             = re.compile(r'\bing\b')
RE_ING2             = re.compile(r'\ning')
RE_HYPHEN_LINEBREAK = re.compile(r'(\w+)-[\s./|]*\n[\s./|]*?(\w+)', re.UNICODE)
RE_NEWLINES         = re.compile(r'\n+')
RE_SPACES           = re.compile(r'\s+')
RE_SPACE_PERIOD     = re.compile(r'\s+\.(?!\S)')
RE_SPACE_COMMA      = re.compile(r'\s+,')
RE_SPACE_PUNCT      = re.compile(r'\s+([!?;:])')    # space before ! ? ; :

def extract_lccn(batch):
    return {'lccn': [
        aid.split('_')[3] if len(aid.split('_')) > 3 else None
        for aid in batch['article_id']
    ]}

@lru_cache(maxsize=20000)
def check_and_correct_word(word):
    base = word.strip(string.punctuation)
    if not base or base.lower() in sym_spell.words:
        return word
    s = sym_spell.lookup(base, Verbosity.CLOSEST, max_edit_distance=1,
                         include_unknown=True, transfer_casing=True)[0].term
    return word.replace(base, s)

def clean_text(text):
    if not isinstance(text, str) or not text.strip():
        return None
    # Character-level normalization (before anything else)
    text = text.translate(_LIGATURE_TABLE)
    text = RE_DOUBLE_QUOTE.sub('"', text)
    # Token-level OCR corrections
    text = RE_DOT_HYPHEN.sub('. -', text)
    text = RE_AUD.sub('and', text)
    text = RE_THO.sub('the', text)
    text = RE_ING1.sub('ing', text)
    text = RE_ING2.sub('ing', text)
    for pat, rep in _TL_LI_SUBS:
        text = pat.sub(rep, text)
    # Line-join heuristic
    text = RE_HYPHEN_LINEBREAK.sub(r'\1\2', text)
    lines = [line.split() for line in text.splitlines()]
    for i in range(len(lines) - 1):
        if not lines[i] or not lines[i + 1]:
            continue
        last, first = lines[i][-1], lines[i + 1][0]
        if last and last[-1] in '.!?;:,':
            continue
        joined = last + first
        if last.endswith('-'):
            lines[i][-1] = last[:-1] + first
            lines[i + 1] = lines[i + 1][1:]
        elif joined:
            if last.lower() not in sym_spell.words or first.lower() not in sym_spell.words:
                sugg = sym_spell.lookup(joined, Verbosity.CLOSEST, max_edit_distance=1,
                                        include_unknown=False, transfer_casing=True)
                if sugg and sugg[0].distance <= 2:
                    sw = sugg[0].term
                    if last and last[0].isupper():
                        sw = sw.capitalize()
                    lines[i][-1] = (sw
                        + ''.join(c for c in last if c in string.punctuation)
                        + ''.join(c for c in first if c in string.punctuation))
                    lines[i + 1] = lines[i + 1][1:]
    text = ' '.join(' '.join(check_and_correct_word(w) for w in line) for line in lines)
    # Final whitespace / punctuation cleanup
    text = RE_NEWLINES.sub(' ', text)
    text = RE_SPACES.sub(' ', text)
    text = RE_SPACE_PERIOD.sub('.', text)
    text = RE_SPACE_COMMA.sub(',', text)
    text = RE_SPACE_PUNCT.sub(r'\1', text)
    return text.strip()

def clean_text_batch(batch):
    return {'article': [clean_text(t) for t in batch['article']]}

def extract_metrics(batch, ocr_detection_segment=7,
                    output_ocr_quality='ocr_quality', output_nonchar='nonchar'):
    ocr_q, nonchar = [], []
    for aid, txt in zip(batch['article_id'], batch['article']):
        if not HAS_OCROSCOPE or txt is None or len(txt.split()) <= ocr_detection_segment:
            ocr_q.append(None); nonchar.append(None); continue
        ev = ocr_evaluation(id=aid, text=txt)
        ev.calculate_ocr_rate(ocr_detection_segment=ocr_detection_segment)
        ocr_q.append(ev.ratio_segment)
        nonchar.append(ev.ratio_nonchar)
    return {output_ocr_quality: ocr_q, output_nonchar: nonchar}

print('Preprocessing functions defined')

In [ ]:
from datasets import load_from_disk
import zipfile

YEARS  = ['1860', '1861', '1862', '1863', '1864', '1865']
N_PROC = os.cpu_count()

def _is_hf_dataset(path):
    """True only if path contains a saved HuggingFace Dataset or DatasetDict."""
    return (os.path.exists(f'{path}/dataset_info.json') or
            os.path.exists(f'{path}/dataset_dict.json'))

def _load_from_hub():
    year_datasets = []
    for year in YEARS:
        print(f'Loading {year}...', end=' ', flush=True)
        ds = load_dataset('dell-research-harvard/AmericanStories', 'subset_years',
                          year_list=[year])[year]
        ds = ds.add_column('year', [int(year)] * len(ds))
        year_datasets.append(ds)
        print(f'{len(ds):,}')
    return concatenate_datasets(year_datasets)

def _extract_zip(zip_path):
    os.makedirs(CORPUS_PATH, exist_ok=True)
    print(f'Extracting {zip_path} into {CORPUS_PATH} (ephemeral) ...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(CORPUS_PATH)
    print(f'Extracted. Corpus at: {CORPUS_PATH}')

def _load_from_local():
    if _is_hf_dataset(CORPUS_PATH):
        print(f'[OK] Local corpus found: {CORPUS_PATH}')
        return load_from_disk(CORPUS_PATH)

    drive_zip = f'{DRIVE_BASE}/americanstories_civilwar.zip'
    if USE_DRIVE and os.path.exists(drive_zip):
        print(f'[OK] Zip found on Drive: {drive_zip}')
        _extract_zip(drive_zip)
    else:
        if USE_DRIVE:
            print(f'Also checked for zip at: {drive_zip}')
        print(f'\nMISSING: no corpus found at {CORPUS_PATH}')
        print('Opening file picker — upload americanstories_civilwar.zip')
        print('(create the zip by running: zip -r americanstories_civilwar.zip americanstories_civilwar/)')
        from google.colab import files as colab_files
        uploaded = colab_files.upload()
        for fname, data in uploaded.items():
            if fname.endswith('.zip'):
                zip_path = f'/content/{fname}'
                with open(zip_path, 'wb') as fh:
                    fh.write(data)
                _extract_zip(zip_path)
                os.remove(zip_path)

    return load_from_disk(CORPUS_PATH)

if USE_LOCAL_CORPUS:
    news_raw = _load_from_local()
else:
    try:
        news_raw = _load_from_hub()
    except Exception as e:
        print(f'HuggingFace download failed: {e}')
        news_raw = _load_from_local()

print(f'Total raw articles: {len(news_raw):,}')

In [ ]:
import time

# ── LCCN extraction + filter ──────────────────────────────────────────────────
t0 = time.time()
print('Extracting LCCNs...')
news_raw = news_raw.map(
    extract_lccn, batched=True, batch_size=4_000, num_proc=N_PROC,
    desc='Extracting LCCNs')
n_before = len(news_raw)
news_raw = news_raw.filter(
    lambda x: x['lccn'] in include_lccns, num_proc=N_PROC,
    desc='Filtering by LCCN whitelist')
print(f'LCCN filter: {len(news_raw):,} / {n_before:,}  ({time.time()-t0:.0f}s)')

# ── Pre-cleaning OCR metrics ──────────────────────────────────────────────────
if HAS_OCROSCOPE:
    t0 = time.time()
    print('Computing pre-cleaning OCR metrics...')
    news_raw = news_raw.map(
        extract_metrics, batched=True, batch_size=2_000, num_proc=N_PROC,
        fn_kwargs={'output_ocr_quality': 'ocr_quality_raw', 'output_nonchar': 'nonchar_raw'},
        desc='OCR metrics (pre-clean)')
    print(f'Pre-clean metrics done  ({time.time()-t0:.0f}s)')

# ── OCR correction ────────────────────────────────────────────────────────────
t0 = time.time()
print('OCR text cleaning (slow — 1-2 hrs for full corpus)...')
news_raw = news_raw.map(
    clean_text_batch, batched=True, batch_size=2_000, num_proc=N_PROC,
    desc='SymSpell OCR correction')
n_before = len(news_raw)
news_raw = news_raw.filter(
    lambda x: x['article'] is not None, num_proc=N_PROC,
    desc='Dropping null articles')
print(f'Non-null after cleaning: {len(news_raw):,} / {n_before:,}  ({time.time()-t0:.0f}s)')

# ── Post-cleaning OCR metrics ─────────────────────────────────────────────────
if HAS_OCROSCOPE:
    t0 = time.time()
    print('Computing post-cleaning OCR metrics...')
    news_raw = news_raw.map(
        extract_metrics, batched=True, batch_size=2_000, num_proc=N_PROC,
        fn_kwargs={'output_ocr_quality': 'ocr_quality_proc', 'output_nonchar': 'nonchar_proc'},
        desc='OCR metrics (post-clean)')
    print(f'Post-clean metrics done  ({time.time()-t0:.0f}s)')

# ── Quality filter (disabled — metrics retained in dataset for later use) ─────
# n_before = len(news_raw)
# news_proc = news_raw.filter(
#     lambda x: (x['ocr_quality_proc'] is not None and x['ocr_quality_proc'] >= 95
#                and x['nonchar_proc']  is not None and x['nonchar_proc']  <= 10),
#     num_proc=N_PROC,
#     desc='Quality filter (ocr_quality>=95, nonchar<=10)')
# print(f'Quality filter: {len(news_proc):,} / {n_before:,} pass')

news_proc = news_raw
print(f'Articles retained (no quality filter): {len(news_proc):,}')

## Section 5a: LSD Sentiment

Lexicoder Sentiment Dictionary — the standard method for political text sentiment
in political science. Outputs 9 metrics per article including proportion-based
and log-odds measures. Runs on CPU; no GPU needed.

In [ ]:
import time

if RUN_LSD:
    def _lsd_batch(batch, lsd_path):
        import re, math, sys
        sys.path.insert(0, lsd_path)
        from lsdSentiment import LexicoderSentimentAnalyzer
        lsd = LexicoderSentimentAnalyzer(
            positive_file     = f'{lsd_path}/positive_words.txt',
            negative_file     = f'{lsd_path}/negative_words.txt',
            neg_positive_file = f'{lsd_path}/neg_positive_words.txt',
            neg_negative_file = f'{lsd_path}/neg_negative_words.txt',
        )
        _CONTRACTION_PATS = [
            (re.compile(r"\b[Aa]in't\b"),  'am not'),
            (re.compile(r"\b[Ll]et's\b"),  'let us'),
            (re.compile(r"\b[Ii]t's\b"),   'it is'),
            (re.compile(r"\b[Ww]on't\b"),  'will not'),
            (re.compile(r"\b[Cc]an't\b"),  'can not'),
            (re.compile(r"\b[Ss]han't\b"), 'shall not'),
            (re.compile(r"n't\b"),           ' not'),
            (re.compile(r"\b[Cc]annot\b"),  'can not'),
            (re.compile(r"'d\b"),             'would'),
            (re.compile(r"'ll\b"),            'will'),
            (re.compile(r"'m\b"),             'am'),
            (re.compile(r"'ve\b"),            'have'),
            (re.compile(r"'re\b"),            'are'),
        ]
        _ABBR_PATS = [
            (re.compile(r' Mr\. '),   ' Mr '),
            (re.compile(r' Mrs\. '),  ' Mrs '),
            (re.compile(r' Ms\. '),   ' Ms '),
            (re.compile(r' Dr\. '),   ' Dr '),
            (re.compile(r' Gen\. '),  ' Gen '),
            (re.compile(r' Gov\. '),  ' Gov '),
            (re.compile(r' Col\. '),  ' Col '),
            (re.compile(r' Maj\. '),  ' Maj '),
            (re.compile(r' Capt\. '), ' Capt '),
            (re.compile(r' Sen\. '),  ' Sen '),
            (re.compile(r' Rep\. '),  ' Rep '),
            (re.compile(r' Jr\. '),   ' Jr '),
            (re.compile(r' Sr\. '),   ' Sr '),
            (re.compile(r' Co\. '),   ' Co '),
        ]
        def _pre(text):
            if not isinstance(text, str): return ''
            for pat, repl in _CONTRACTION_PATS: text = pat.sub(repl, text)
            for pat, repl in _ABBR_PATS:        text = pat.sub(repl, text)
            return text
        def _score(text):
            text = _pre(text)
            r = lsd.analyze(text)
            pos, neg, tot = r['pos_count'], r['neg_count'], max(r['total_words'], 1)
            ps, ns = pos / tot, neg / tot
            return {
                'lsd_pos':         pos,
                'lsd_neg':         neg,
                'lsd_pos_share':   ps,
                'lsd_neg_share':   ns,
                'lsd_neu_share':   1.0 - ps - ns,
                'lsd_relpropdiff': (pos - neg) / tot,
                'lsd_abspropdiff': abs(pos - neg) / tot,
                'lsd_logit':       math.log((pos + 0.5) / (neg + 0.5)),
                'lsd_asinh':       math.asinh((pos - neg) / tot),
            }
        rows = [_score(t) for t in batch['article']]
        return {k: [r[k] for r in rows] for k in rows[0]}

    print(f'Computing LSD for {len(news_proc):,} articles ({N_PROC} workers)...')
    t0 = time.time()
    news_proc = news_proc.map(
        _lsd_batch, batched=True, batch_size=2_000, num_proc=N_PROC,
        fn_kwargs={'lsd_path': LSD_PATH},
        desc='LSD scoring')
    print(f'LSD done  ({time.time()-t0:.0f}s)')
    lsd_cols = [c for c in news_proc.column_names if c.startswith('lsd_')]
    print('LSD columns added:', lsd_cols)
else:
    print('LSD skipped (RUN_LSD=False)')

## Section 5b: VADER Sentiment

In [ ]:
if RUN_VADER:
    def _vader_batch(batch):
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        vader = SentimentIntensityAnalyzer()
        results = [vader.polarity_scores(t if isinstance(t, str) else '')
                   for t in batch['article']]
        return {
            'vader_sent': [r['compound'] for r in results],
            'vader_pos':  [r['pos']      for r in results],
            'vader_neg':  [r['neg']      for r in results],
            'vader_neu':  [r['neu']      for r in results],
        }

    print(f'Computing VADER for {len(news_proc):,} articles ({N_PROC} workers)...')
    t0 = time.time()
    news_proc = news_proc.map(
        _vader_batch, batched=True, batch_size=2_000, num_proc=N_PROC,
        desc='VADER scoring')
    mean_sent = sum(news_proc['vader_sent']) / len(news_proc)
    print(f'VADER done  ({time.time()-t0:.0f}s). Mean compound: {mean_sent:.4f}')
else:
    print('VADER skipped (RUN_VADER=False)')

## Section 6: Save

In [ ]:
import zipfile, time
from huggingface_hub import HfApi

LOCAL_PROC_PATH = f'{LOCAL_BASE}/news_proc'
LOCAL_ZIP_PATH  = '/content/news_proc.zip'

# Save to ephemeral storage (avoids Drive space limits)
print(f'Saving {len(news_proc):,} articles to {LOCAL_PROC_PATH} ...')
print(f'Columns: {news_proc.column_names}')
t0 = time.time()
news_proc.save_to_disk(LOCAL_PROC_PATH)
print(f'Saved  ({time.time()-t0:.0f}s)')

# Zip the saved dataset
print(f'Zipping to {LOCAL_ZIP_PATH} ...')
t0 = time.time()
with zipfile.ZipFile(LOCAL_ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for dirpath, _, filenames in os.walk(LOCAL_PROC_PATH):
        for fname in filenames:
            fpath = os.path.join(dirpath, fname)
            zf.write(fpath, arcname=os.path.relpath(fpath, LOCAL_BASE))
zip_mb = os.path.getsize(LOCAL_ZIP_PATH) / 1e6
print(f'Zip complete: {zip_mb:.1f} MB  ({time.time()-t0:.0f}s)')

# Upload to HF
_api = HfApi()
_api.create_repo(HF_REPO, private=True, repo_type='dataset', exist_ok=True)
try:
    _existing = list(_api.list_repo_files(HF_REPO, repo_type='dataset'))
except Exception:
    _existing = []
if 'news_proc.zip' in _existing:
    print('[already on HF] news_proc.zip')
else:
    print(f'Uploading news_proc.zip to {HF_REPO} ({zip_mb:.1f} MB) ...')
    _api.upload_file(
        path_or_fileobj=LOCAL_ZIP_PATH,
        path_in_repo='news_proc.zip',
        repo_id=HF_REPO,
        repo_type='dataset',
    )
    print(f'Saved: hf://datasets/{HF_REPO}/news_proc.zip')

print('Notebook 1 complete.')

## Checkpoint — reload after session restart

Run this cell on subsequent sessions to reload the saved dataset
(`USE_DRIVE=True` only — ephemeral `/content/` does not survive restarts).


In [ ]:
from datasets import load_from_disk
news_proc1 = load_from_disk(NEWS_PROC_PATH)
print(f'Loaded {len(news_proc1):,} articles from {NEWS_PROC_PATH}')
print('Columns:', news_proc1.column_names)

In [ ]:
news_proc1[1000000]